# Week 5 skills notebook: averages, spread and the shape of data

This notebook belongs to the skills track. Every table in it is made up with a fixed seed, so the skills stay separate from this week's project and your numbers match everyone else's.

You will draw the shape of four processes, compute two kinds of centre, play a game until its average settles, watch a favourable bet ruin most of its players, measure spread, catch an AI that compared two groups by their averages alone, and see why a small shift in an average can change the extremes a lot.

**How to use it:** run each cell in order with the play button or Shift+Enter. Read the output before you move on. Three "your turn" tasks each end with a CHECK cell.

## 1. A distribution is the shape of a process

A distribution records how often each value occurs. Different processes leave different shapes. The cell below makes 10,000 values from each of four processes:

- the minute past the hour at which a phone call arrives,
- the weight in grams of a 1 kg bag filled by a machine,
- the minutes between two buses,
- a monthly income, in a made-up currency.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(51)
n = 10_000
shapes = pd.DataFrame({
    "call_minute": rng.uniform(0, 60, n),                     # any minute is as likely as any other
    "bag_weight_g": rng.normal(1000, 8, n),                    # many small errors added together
    "bus_gap_min": rng.exponential(10, n),                     # waiting for something that can happen at any moment
    "monthly_income": rng.lognormal(np.log(3000), 0.8, n),    # growth that multiplies
})
for col in shapes:
    s = shapes[col]
    print(f"{col}: mean {s.mean():.1f}, median {s.median():.1f}, standard deviation {s.std():.1f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, col in zip(axes.flat, shapes.columns):
    ax.hist(shapes[col], bins=60)
    ax.set_title(col)
    ax.set_ylabel("number of values")
plt.tight_layout()
plt.show()

Look at the four histograms before you read on. The call minutes are flat. The bag weights form a bell. The bus gaps start high and decay. The incomes rise quickly and then trail off to the right for a long way.

**The tail is where surprises live.** The tail is the far end of a distribution. The next cell counts how often each process produces a value more than three standard deviations above its mean. (The standard deviation measures the typical distance from the mean. Section 6 computes it by hand.)

In [ ]:
print("share of values more than 3 standard deviations above the mean:")
for col in shapes:
    s = shapes[col]
    far = (s > s.mean() + 3 * s.std()).mean()
    print(f"  {col}: {far:.2%}")

## 2. Sums make bells

Why does the bell appear so often? The total of many independent small effects takes the shape of a bell, whatever the shape of each effect. One die is flat. Watch what happens to the total of two dice and of ten dice.

In [ ]:
rng = np.random.default_rng(52)
one = rng.integers(1, 7, size=10_000)
two = rng.integers(1, 7, size=(10_000, 2)).sum(axis=1)
ten = rng.integers(1, 7, size=(10_000, 10)).sum(axis=1)
print("most common total of two dice:", pd.Series(two).mode()[0])

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, values, name in zip(axes, [one, two, ten], ["one die", "total of two dice", "total of ten dice"]):
    ax.hist(values, bins=np.arange(values.min(), values.max() + 2) - 0.5)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 3. Two centres: the mean and the median

The **mean** adds every value and divides by how many there are. The **median** is the middle value once the values are sorted, so half lie below it and half above.

A small office has six staff and one manager. Compute both centres by hand first, then let pandas confirm them.

In [ ]:
pay = pd.Series([2400, 2500, 2600, 2700, 2800, 3000, 14000])   # monthly pay: six staff, then the manager
print("mean pay:", round(pay.sum() / len(pay), 2))
print("median pay:", pay.sort_values().iloc[len(pay) // 2])
print("pandas agrees:", round(pay.mean(), 2), pay.median())

One large value pulled the mean far above what six of the seven people earn, and it left the median where it was. The same thing happens across the 10,000 made-up incomes from section 1.

In [ ]:
income = shapes["monthly_income"]
print(f"mean income: {income.mean():.0f}")
print(f"median income: {income.median():.0f}")
print(f"share of people who earn less than the mean: {(income < income.mean()).mean():.1%}")

## 4. The long-run average

The **expected value** of a random outcome is the average you would get if you repeated it many times. Multiply each possible outcome by its probability and add the results.

A game pays three times the roll of one die. Compute its expected prize, then play it 5,000 times and watch the running average settle.

In [ ]:
prizes = np.arange(1, 7) * 3            # the six possible prizes
print("possible prizes:", prizes.tolist())
print("expected prize:", (prizes * (1 / 6)).sum())

rng = np.random.default_rng(54)
played = rng.integers(1, 7, size=5000) * 3
running = played.cumsum() / np.arange(1, 5001)
for k in [10, 100, 1000, 5000]:
    print(f"average prize after {k} games: {running[k - 1]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(1, 5001), running)
ax.axhline((prizes / 6).sum(), color="grey", linestyle="--")
ax.set_xscale("log")
ax.set_title("Average prize so far (dashed line: the expected prize)")
ax.set_xlabel("games played (log scale)")
ax.set_ylabel("average prize")
plt.tight_layout()
plt.show()

A ticket priced at the expected prize is fair: over many games, the player neither gains nor loses on average. Early on, the running average wanders. The long run is long.

## 5. A favourable bet that ruins most players

Each round, a player stakes everything on a coin toss. Heads multiplies the money by 1.5. Tails multiplies it by 0.6. The expected change per round is a gain, so the bet looks good. Play it for 10,000 players over 40 rounds.

In [ ]:
rng = np.random.default_rng(55)
players, rounds, start = 10_000, 40, 100
heads = rng.random((players, rounds)) < 0.5
wealth = start * np.where(heads, 1.5, 0.6).prod(axis=1)     # heads: times 1.5, tails: times 0.6
print("players:", players, "rounds:", rounds, "starting money:", start)
print("expected change per round: times", 0.5 * 1.5 + 0.5 * 0.6)
print(f"mean final money: {wealth.mean():.1f}")
print(f"median final money: {np.median(wealth):.1f}")
print(f"share of players who end with less than they started: {(wealth < start).mean():.1%}")

The mean is high because a few players got very lucky. The median player has lost most of the money. The expected value describes the average over many players, and no single player gets to live that average.

**Rule of thumb:** trust the mean for a decision that is repeated many times by someone who can absorb the losses on the way. Distrust it when one bad run can end the game.

## 6. Spread: the same mean, different worlds

The **variance** is the average squared distance from the mean. The **standard deviation** is its square root, so it is in the same unit as the data and reads as the typical distance from the mean.

Compute both by hand for five waiting times. There is one trap: software divides the sum of squared distances either by the number of values (n) or by one less (n - 1). pandas uses n - 1, and numpy uses n.

In [ ]:
w = pd.Series([20, 25, 30, 35, 40])            # five waits, in minutes
dist = w - w.mean()
print("distances from the mean:", dist.tolist())
print("variance, dividing by n:", (dist ** 2).sum() / len(w))
print("variance, dividing by n - 1:", (dist ** 2).sum() / (len(w) - 1))
print("pandas w.var():", w.var(), "  numpy np.var(w):", np.var(w.to_numpy()))
print("standard deviation, dividing by n:", round(np.sqrt((dist ** 2).sum() / len(w)), 2))

With five values, the two ways of dividing give clearly different answers. With thousands of values they agree to the second decimal. Whichever you use, say which one it is.

**Percentiles** describe spread without assuming any shape. The 90th percentile is the value that 90% of the values lie below. The distance from the 10th to the 90th percentile holds the middle 80% of the values.

Now meet two clinics. Each records the waiting time of 2,000 patients.

In [ ]:
rng = np.random.default_rng(56)
waits = pd.DataFrame({
    "clinic": ["A"] * 2000 + ["B"] * 2000,
    "wait_min": np.concatenate([rng.normal(30, 5, 2000),           # clinic A: everyone waits about the same
                                rng.exponential(30, 2000)]).round(1),   # clinic B: most are quick, a few wait very long
})
print(waits.groupby("clinic").size())
print(waits.sample(6, random_state=1))

## 7. Read code an AI assistant wrote

We asked an AI assistant: *"Do patients wait longer at clinic A or at clinic B?"* It wrote the cell below. Run it, then decide whether the conclusion is right.

In [ ]:
# Written by an AI assistant
avg = waits.groupby("clinic")["wait_min"].mean()
print(f"Clinic A: {avg['A']:.1f} minutes on average")
print(f"Clinic B: {avg['B']:.1f} minutes on average")
diff = abs(avg["A"] - avg["B"])
print(f"The difference is {diff:.1f} minutes, so patients wait about the same time at both clinics.")

The arithmetic is right, and the conclusion is still wrong. The assistant reduced each clinic to one number and never looked at the shape. A patient does not experience the average wait. A patient experiences one wait, and the question is how long it might be.

The next two cells show what the averages hide.

In [ ]:
for clinic, s in waits.groupby("clinic")["wait_min"]:
    print(f"clinic {clinic}: median {s.median():.1f}, standard deviation {s.std():.1f}, "
          f"more than 60 minutes {(s > 60).mean():.1%}, longest {s.max():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
bins = np.arange(0, waits["wait_min"].max() + 5, 5)
for clinic, s in waits.groupby("clinic")["wait_min"]:
    ax.hist(s, bins=bins, alpha=0.6, label=f"clinic {clinic}")
ax.axvline(60, color="grey", linestyle="--")
ax.set_title("Waiting times at two clinics with the same average (dashed line: one hour)")
ax.set_xlabel("wait in minutes")
ax.set_ylabel("patients")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Your turn: a summary that does not hide the shape

Ask your AI assistant for code that makes a table called `summary` with one row per clinic (the index is `"A"` and `"B"`) and five columns: `mean`, `median`, `sd`, `p90` (the 90th percentile) and `share_over_60` (the share of patients who waited more than 60 minutes, as a fraction between 0 and 1).

Paste its code into the cell below, run it, and then run the CHECK cell. Before you trust the table, compare one row with the output of section 7.

In [ ]:
# your turn 1: make a table called summary with one row per clinic and the columns mean, median, sd, p90 and share_over_60
summary = None

In [ ]:
# CHECK 1
assert summary is not None, "summary is still empty"
for clinic, s in waits.groupby("clinic")["wait_min"]:
    row = summary.loc[clinic]
    assert abs(row["mean"] - s.mean()) < 0.01, f"mean for clinic {clinic}"
    assert abs(row["median"] - s.median()) < 0.01, f"median for clinic {clinic}"
    assert abs(row["sd"] - s.std()) < 0.05, f"sd for clinic {clinic}"         # n or n - 1 both pass with 2,000 values
    assert abs(row["p90"] - s.quantile(0.9)) < 0.5, f"p90 for clinic {clinic}"
    assert abs(row["share_over_60"] - (s > 60).mean()) < 0.001, f"share_over_60 for clinic {clinic} (a fraction, not a percentage)"
print("CHECK 1 passed")
print(summary.round(3))

## 9. The average of many is steadier than one

Single waits at clinic B spread widely. What about the average of 16 waits, or of 256? The cell below draws 5,000 averages for each size and measures how much they spread.

In [ ]:
rng = np.random.default_rng(57)
print(f"standard deviation of single waits: {rng.exponential(30, 1_000_000).std():.1f}")
for k in [1, 4, 16, 64, 256]:
    averages = rng.exponential(30, size=(5000, k)).mean(axis=1)    # 5,000 averages, each of k waits
    print(f"n = {k}: standard deviation of the averages {averages.std():.2f}, square-root rule 30 / sqrt({k}) = {30 / np.sqrt(k):.2f}")

In [ ]:
rng = np.random.default_rng(58)
fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharex=True)
for ax, k in zip(axes, [1, 16, 256]):
    ax.hist(rng.exponential(30, size=(5000, k)).mean(axis=1), bins=60)
    ax.set_title("single waits" if k == 1 else f"averages of {k} waits")
    ax.set_xlabel("minutes")
plt.tight_layout()
plt.show()

**The square-root rule:** the spread of an average equals the spread of single values divided by the square root of how many values it averages. Four times as much data halves the spread of the average. The averages also turn into a bell, for the reason section 2 gave: an average is a sum divided by a count.

Two cautions. The rule is about the average, not about single values: clinic B's patients still wait as unpredictably as before. And the rule assumes the values are independent. Values that move together, such as the sales of one shop on days in a row, shrink the spread of their average more slowly.

## 10. Your turn: test the rule on clinic A

Clinic A's waits follow a bell with a mean of 30 minutes and a standard deviation of 5 minutes. Ask your AI assistant for code that makes a pandas Series called `spread_of_avg`, indexed by n = 1, 4, 16 and 64, holding the standard deviation of 5,000 averages of n waits drawn with `rng.normal(30, 5, ...)`.

Before you run the CHECK, write down what the square-root rule predicts for each n.

In [ ]:
# your turn 2: make a Series called spread_of_avg, indexed by n = 1, 4, 16, 64
spread_of_avg = None

In [ ]:
# CHECK 2
assert spread_of_avg is not None, "spread_of_avg is still empty"
for k in [1, 4, 16, 64]:
    rule = 5 / np.sqrt(k)
    assert abs(spread_of_avg[k] - rule) / rule < 0.05, f"n = {k} is more than 5% away from the square-root rule"
print("CHECK 2 passed:", spread_of_avg.round(2).to_dict())

## 11. A small shift in the average, a large change in the tail

A building's daily electricity use follows a bell. Call a day "high-use" when it is above the old 90th percentile, so one day in ten is high-use. Now every day uses a little more. Watch the average and the high-use days.

In [ ]:
rng = np.random.default_rng(59)
before = rng.normal(50, 10, 100_000)          # daily use in kWh
threshold = np.percentile(before, 90)          # a high-use day is above the old 90th percentile
after = before + 3                             # every day uses 3 kWh more
wider = 53 + 1.2 * (before - 50)               # 3 kWh more, and the spread 20% wider
print(f"high-use threshold: {threshold:.1f} kWh")
print(f"the average rose by {(after.mean() - before.mean()) / before.mean():.1%}")
print(f"high-use days before: {(before > threshold).mean():.1%}")
print(f"high-use days after the shift: {(after > threshold).mean():.1%}")
print(f"high-use days after the shift with a wider spread: {(wider > threshold).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
bins = np.arange(0, 101, 1)
ax.hist(before, bins=bins, alpha=0.5, density=True, label="before")
ax.hist(wider, bins=bins, alpha=0.5, density=True, label="after, with a wider spread")
ax.axvline(threshold, color="grey", linestyle="--")
ax.set_title("Daily electricity use (dashed line: the high-use threshold)")
ax.set_xlabel("kWh per day")
ax.set_ylabel("share of days per kWh")
ax.legend()
plt.tight_layout()
plt.show()

Many days sat just below the threshold. A small push moves them over it, so the share above the threshold grows much faster, in relative terms, than the average does. When a question is about extremes, measure the extremes directly.

## 12. Your turn: how fast does the tail grow?

Ask your AI assistant for code that makes a dictionary called `shares`. It maps each shift of 1, 2 and 5 kWh to the share of days above `threshold` after every value in `before` is raised by that shift. Predict the order of the three shares before you run the CHECK.

In [ ]:
# your turn 3: make a dict called shares that maps each shift (1, 2 and 5) to the share of days above threshold
shares = None

In [ ]:
# CHECK 3
assert shares is not None, "shares is still empty"
for shift in [1, 2, 5]:
    assert abs(shares[shift] - (before + shift > threshold).mean()) < 1e-9, f"the share for a shift of {shift}"
assert shares[1] < shares[2] < shares[5]
print("CHECK 3 passed:", {k: round(float(v), 3) for k, v in shares.items()})
print("a shift of 5 kWh makes high-use days", round(shares[5] / (before > threshold).mean(), 2), "times as common")

## 13. What to carry forward

Before you compare two sets of numbers, work through four steps.

1. Draw both histograms on the same bins.
2. Choose a centre that fits the shape: the median for a long tail, the mean for a bell.
3. Report a spread next to the centre: a standard deviation, or the 10th and 90th percentiles.
4. If the question is about extremes, count the extremes directly with a threshold chosen before you look at the result.